# Cuaderno de entrenamiento del modelo



## Configuración del cuaderno


In [ ]:
#Instalación de paquetes
!pip install tf_keras tensorflow numpy matplotlib -q
!pip install coral-ordinal

import os

# Forzado de Keras 2 para evitar problemas de compatibilidad con STM32Cube.AI
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras
from tf_keras import layers, Model
import coral_ordinal as coral

# Verificar que estamos en Keras 2
assert int(keras.__version__.split('.')[0]) == 2, "No se está usando la versión de Keras2"
print("Usando Keras2")

from google.colab import drive
drive.mount('/content/drive')

import sys

SRC_PATH = "/content/drive/MyDrive/TFG/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

In [ ]:
# Macros

TRAIN_PATH = '/content/drive/MyDrive/TFG/dataset/train'
VAL_PATH = '/content/drive/MyDrive/TFG/dataset/val'

CHECKPOINT_PATH = '/content/drive/MyDrive/TFG/checkpoints'
MODEL_PATH = '/content/drive/MyDrive/TFG/modelos'

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

IMAGE_SIZE = (480, 270)
INPUT_SHAPE = (224, 224, 3)

NUM_BATCHES = 32
NUM_EPOCH = 50

LABELS = {'0':'fluido', '1' : 'moderado', '2' : 'denso', '3' : 'saturado'}
NUM_CLASES = len(LABELS)


## Funciones auxiliares

In [ ]:
# Preprocesado de imágenes
from preprocesado import preprocesado
preprocesado = preprocesado(IMAGE_SIZE)

## Dataset

In [ ]:
# Carga del dataset
train_ds = keras.utils.image_dataset_from_directory (
    directory = TRAIN_PATH,
    labels = 'inferred',
    label_mode = 'int',
    batch_size = NUM_BATCHES,
    image_size = IMAGE_SIZE
)

val_ds = keras.utils.image_dataset_from_directory (
    directory = VAL_PATH,
    labels = 'inferred',
    label_mode = 'int',
    batch_size = NUM_BATCHES,
    image_size = IMAGE_SIZE
)

# Preprocesado del dataset
train_ds = train_ds.map(
    lambda x, y: (preprocesado(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.map(
    lambda x, y: (preprocesado(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

## Definición del modelo: MobileNet

In [ ]:
#Definición del modelo
base_model = keras.applications.MobileNet(
    input_shape= INPUT_SHAPE,
    alpha=0.50,
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = keras.Input(shape=INPUT_SHAPE, name='input')
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3, name='dropout')(x)
outputs = coral.CoralOrdinal(num_classes=NUM_CLASES, name='output')(x)

model = Model(inputs, outputs, name='MobileNetV1_a050_coral')

# Compilación del modelo
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=coral.OrdinalCrossEntropy (num_classes=NUM_CLASES),
    metrics=[
        coral.MeanAbsoluteErrorLabels(name='mae_labels')
    ]
)

model.summary()


## Entrenamiento del modelo

In [ ]:
# Definición de los callbacks
callbacks = [

    # Guarda el mejor modelo
    keras.callbacks.ModelCheckpoint(
        filepath        = f'{CHECKPOINT_PATH}/best_model.keras',
        monitor         = 'val_loss',
        save_best_only  = True,
        verbose         = 1
    ),

    # Early stopping
    keras.callbacks.EarlyStopping(
        monitor              = 'val_loss',
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),

    # Reduce LR
    keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 3,
        min_lr   = 1e-6,
        verbose  = 1
    )

]

# Entrenamiento
history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = NUM_EPOCH,
    callbacks       = callbacks
)

# Guardado de los resultados de entrenamiento
import json

with open(f'{MODEL_PATH}/history.json', 'w') as f:
    json.dump(
        {k: [float(v) for v in vals] for k, vals in history.history.items()},
        f,
        indent=2
    )

model.save(f'{MODEL_PATH}/modelo_final.keras')
print("Modelo guardado en:", MODEL_PATH)